# A FAIR² Mental Health Survey Dataset from Kilifi County, Kenya: Demonstrating AI-Ready Data Standards for Data Infrastructure in Africa Exploration with `mlcroissant`
This notebook provides a step-by-step guide for loading, exploring, and analyzing the [FAIR² Mental Health Survey Dataset from Kilifi County, Kenya](https://sen.science/doi/10.71728/senscience.vcs2-05nj) using the `mlcroissant` library, following best practices for referencing all dataset entities by their `@id` fields.

### Dataset Source
The dataset source is provided via a Croissant schema:

`https://sen.science/doi/10.71728/senscience.vcs2-05nj/fair2.json`

In [ ]:
# Ensure `mlcroissant` and `pandas` are installed
!pip install mlcroissant --quiet
!pip install pandas --quiet

## 1. Data Loading
Load metadata and records from the dataset using `mlcroissant`.

In [ ]:
import mlcroissant as mlc
import pandas as pd
import json

# Define the dataset URL
croissant_url = 'https://sen.science/doi/10.71728/senscience.vcs2-05nj/fair2.json'

# Load the dataset metadata
ds = mlc.Dataset(croissant_url)
print(f"{ds.metadata.name}: {ds.metadata.description}\n")

## 2. Data Overview
List available record sets and their fields, referencing all by their `@id` values, as per the Croissant schema. This helps guide selection for extraction and analysis later.

In [ ]:
# Display available record sets and their fields by `@id`
record_sets = ds.record_sets
print("Available record sets (by @id):\n")
for rs in record_sets:
    print(f"- Record Set @id: {rs['@id']}, name: {rs['name']}")
    print("  Fields:")
    for f in rs['fields']:
        print(f"    - Field @id: {f['@id']} | name: {f.get('name', '')} | Data type: {f.get('dataType', '')}")
    print()

## 3. Data Extraction
Load data from a specific record set into a DataFrame for analysis, referencing record set and field `@id`s from the overview above.

In [ ]:
# Collect all record set @id values
record_set_ids = [rs['@id'] for rs in record_sets]
dataframes = {}

for record_set_id in record_set_ids:
    records = list(ds.records(record_set=record_set_id))
    if records:
        df = pd.DataFrame(records)
        dataframes[record_set_id] = df
        print(f"Loaded {len(df)} rows for Record Set @id: {record_set_id} with columns: {df.columns.tolist()}")
    else:
        print(f"No records found for Record Set @id: {record_set_id}")

# Select main record set for demonstration (first, usually the main one)
if dataframes:
    main_record_set_id = list(dataframes.keys())[0]
    print(f"\nExample preview from record set @id: {main_record_set_id}")
    display(dataframes[main_record_set_id].head())
else:
    main_record_set_id = None
    print("No dataframes loaded!")

## 4. Exploratory Data Analysis (EDA)
Demonstrate common data processing tasks: filtering on a numeric field, normalization, and grouping by a categorical field. All field references are by `@id` as shown in the Overview.

In [ ]:
# Adapt these variables to reference your fields by their @id

# You may need to inspect field IDs in previous cells to set these correctly:
# For demonstration, let's search for a likely numeric field (e.g. GAD-7 score, PHQ-9 score, etc.)

if main_record_set_id is not None:
    df = dataframes[main_record_set_id]
    # Try auto-detect a numeric field (or manually specify here if known):
    numeric_candidate_ids = [col for col in df.columns if any(x in col.lower() for x in ['gad', 'phq', 'psq', 'score', 'age'])]
    if numeric_candidate_ids:
        # Use the first matching as demo
        numeric_field_id = numeric_candidate_ids[0]
        print(f"Using numeric field for analysis: {numeric_field_id}\n")
        # Ensure numeric dtype
        df[numeric_field_id] = pd.to_numeric(df[numeric_field_id], errors='coerce')
        # Set threshold to mean for demonstration
        threshold = df[numeric_field_id].mean()
        filtered_df = df[df[numeric_field_id] > threshold].copy()
        print(f"Filtered records with {numeric_field_id} > {threshold:.2f} (N={len(filtered_df)})")
        display(filtered_df.head())

        norm_col = f"{numeric_field_id}_normalized"
        filtered_df[norm_col] = (filtered_df[numeric_field_id] - filtered_df[numeric_field_id].mean()) / filtered_df[numeric_field_id].std()
        print(f"Normalized {numeric_field_id} for filtered records:")
        display(filtered_df[[numeric_field_id, norm_col]].head())

        # Try to pick a likely categorical variable for grouping (e.g. gender/sex, education, etc.)
        group_candidate_ids = [col for col in df.columns if any(x in col.lower() for x in ['gender', 'sex', 'marital', 'village', 'education'])]
        if group_candidate_ids:
            group_field_id = group_candidate_ids[0]
            grouped_df = filtered_df.groupby(group_field_id)[numeric_field_id].mean().reset_index()
            print(f"Grouped mean {numeric_field_id} by {group_field_id}:")
            display(grouped_df.head())
        else:
            print("No suitable group_field found for demonstration.")
    else:
        print("No suitable numeric field found for demonstration.")
else:
    print("No data was loaded in the previous step.")

## 5. Visualization
Visualize the distribution of the selected numeric field and the group comparison, if available.

In [ ]:
import matplotlib.pyplot as plt
import seaborn as sns

if main_record_set_id is not None and 'numeric_field_id' in locals():
    # Distribution plot
    fig, ax = plt.subplots(figsize=(7, 4))
    sns.histplot(df[numeric_field_id].dropna(), bins=20, kde=True, ax=ax)
    ax.set_title(f'Distribution of {numeric_field_id}')
    plt.xlabel(numeric_field_id)
    plt.ylabel('Count')
    plt.show()

    # If grouping was possible, visualize grouped means
    if 'grouped_df' in locals():
        fig, ax = plt.subplots(figsize=(8, 4))
        sns.barplot(data=grouped_df, x=group_field_id, y=numeric_field_id, ax=ax)
        ax.set_title(f'Mean {numeric_field_id} by {group_field_id}')
        plt.xticks(rotation=45)
        plt.show()
else:
    print("No numeric field available or no data loaded; cannot plot.")

## 6. Conclusion
This notebook demonstrated how to:
- Load Croissant metadata and records using the `mlcroissant` Python library.
- Reference all dataset entities using their Croissant `@id` for robustness and clarity.
- Extract tabular data by record set for deeper analysis in pandas.
- Perform standard exploratory data analysis with normalization and grouping.
- Visualize numeric data fields and group statistics when relevant.

**By referencing all columns, fields, and record sets by `@id`, the analysis remains reproducible and unambiguous, even when dataset schemas are updated.**

Explore more: check the [mlcroissant reference documentation](https://mlcommons.org/croissant/) for advanced extraction and interoperability features.